# Hybrid Mamba-xLSTM: Complete End-to-End Validation

**Objective**: Validate full training pipeline on Colab T4 before A100 production  
**Branch**: `a100_70m_baseline` from https://github.com/krishankb-de/hybrid_model_mamba_xlstm
**Model**: Hybrid 70M | Dataset: PubMed sample | Time: ~2.5-3 hours


## PART 1: GPU & Environment Check

In [ ]:
import subprocess, sys
print('=== GPU Info ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print(f'Python {sys.version}')

## PART 2: Clone a100_70m_baseline & Install

In [ ]:
import os, subprocess
repo_dir = '/content/hybrid_model_mamba_xlstm'
if not os.path.exists(repo_dir):
    result = subprocess.run(['git', 'clone', '--branch', 'a100_70m_baseline',
        'https://github.com/krishankb-de/hybrid_model_mamba_xlstm.git', repo_dir],
        capture_output=True, text=True)
    print(result.stdout)
else:
    os.chdir(repo_dir)
    result = subprocess.run(['git', 'status'], capture_output=True, text=True)
    print(result.stdout)
os.chdir(repo_dir)

In [ ]:
import subprocess, os
os.chdir('/content/hybrid_model_mamba_xlstm')
print('Installing package...')
result = subprocess.run(['pip', 'install', '-e', '.'], capture_output=True, text=True, timeout=300)
print('✅ Package installed')

## PART 3: Run Unit Tests

In [ ]:
import subprocess, os
os.chdir('/content/hybrid_model_mamba_xlstm')
result = subprocess.run(['pytest', 'tests/test_encoder_pooling.py', '-v', '-x'],
    capture_output=True, text=True, timeout=60)
print(result.stdout)
print('✅ Unit tests passed' if result.returncode == 0 else '❌ Tests failed')

## PART 4: Stage 0 Vanilla LM (500 steps, ~30-40 min)

In [ ]:
import subprocess, os
os.chdir('/content/hybrid_model_mamba_xlstm')
cmd = ['python', 'scripts/train.py', 'model=hybrid_70m', 'dataset=pubmed',
    'trainer=a100_single_gpu', 'trainer.batch_size=8', 'trainer.max_steps=500',
    'experiment_name=colab_stage0_vanilla']
result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
print(result.stdout)
print('\n✅ Stage 0 Vanilla Complete')

## PART 5: Stage 1 Vanilla SimCSE (200 steps, ~15-20 min)

In [ ]:
import subprocess, os
os.chdir('/content/hybrid_model_mamba_xlstm')
cmd = ['python', 'scripts/train_contrastive.py', 'model=hybrid_70m', 'dataset=pubmed',
    'trainer=a100_single_gpu', 'trainer.batch_size=16', 'trainer.max_steps=200',
    'checkpoint=outputs/colab_stage0_vanilla/checkpoints/last.ckpt',
    'experiment_name=colab_stage1_vanilla']
result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
print(result.stdout)
print('\n✅ Stage 1 Vanilla Complete')

## PART 6: Validation Report

In [ ]:
import os, json
os.chdir('/content/hybrid_model_mamba_xlstm')
checkpoints = {
    'S0 Vanilla': 'outputs/colab_stage0_vanilla/checkpoints/last.ckpt',
    'S1 Vanilla': 'outputs/colab_stage1_vanilla/checkpoints/last.ckpt'
}
print('=== CHECKPOINT STATUS ===')
for name, path in checkpoints.items():
    if os.path.exists(path):
        print(f'{name}: ✅ READY ({os.path.getsize(path)/(1024**2):.0f}MB)')
    else:
        print(f'{name}: ⏭️ SKIPPED')
print('\n✅ Validation complete - Ready for download')

## GITHUB WORKFLOW: Next Steps After Colab

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║  GITHUB WORKFLOW: Next Steps After Colab Completes           ║
╚══════════════════════════════════════════════════════════════╝

STEP 1: ON A100 SERVER - Pull Latest Code
──────────────────────────────────────────
$ cd /path/to/hybrid_model_mamba_xlstm
$ git fetch origin
$ git checkout a100_70m_baseline
$ git pull origin a100_70m_baseline

STEP 2: Download Colab Checkpoints
───────────────────────────────────
# From Colab: Download
#   - outputs/colab_stage0_vanilla/checkpoints/last.ckpt
#   - outputs/colab_stage1_vanilla/checkpoints/last.ckpt

# To A100:
$ scp -r colab_ckpts/* user@a100:/path/to/hybrid_model/colab_checkpoints/

STEP 3: Verify Checkpoint Compatibility
────────────────────────────────────────
$ python3 check_checkpoint_compatibility.py \\
    --checkpoint colab_checkpoints/last.ckpt

Expected: ✅ Compatible with a100_70m_baseline

STEP 4: Launch A100 Production Training
────────────────────────────────────────
# Stage 0: 40k steps full dataset
$ sbatch scripts/train_stage0_lm_pubmed.sh

# Wait for completion, then Stage 1: 10k steps
# (auto-initializes from Stage 0 checkpoint)
$ sbatch scripts/train_contrastive_stage1.sh

Total time: ~10-15 hours

STEP 5: Monitor Progress
────────────────────────
$ tail -f logs/train_stage0.log      # Monitor Stage 0
$ watch -n 10 'nvidia-smi'           # GPU usage
$ squeue -u $USER                    # Slurm queue

STEP 6: After Training Completes
─────────────────────────────────
$ python scripts/evaluate_lm.py \\
    --checkpoint outputs/hybrid_70m/checkpoints/last.ckpt \\
    --batch-size 32 --throughput

STEP 7: Commit to GitHub
────────────────────────
$ git add outputs/hybrid_70m/checkpoints/
$ git add evaluation_results/
$ git commit -m "A100 production: stage0 (40k) + stage1 (10k) complete

- PPL: X.XX
- NDCG: 0.XX
- Throughput: XXXX tok/s"
$ git push origin a100_70m_baseline

STEP 8: Create Release Tag (Optional)
──────────────────────────────────────
$ git tag -a v1.0-prod -m "A100 production model (70M hybrid)"
$ git push origin v1.0-prod

╔══════════════════════════════════════════════════════════════╗
║  KEY POINTS                                                  ║
╚══════════════════════════════════════════════════════════════╝

✓ Always pull fresh from a100_70m_baseline before training
✓ Verify checkpoint compatibility before A100 run
✓ Use sbatch (not direct python) for long jobs
✓ Expected A100 time: 10-15 hours (can reduce with larger batch)
✓ Save evaluation results to GitHub for reproducibility
✓ Don't commit large .ckpt files if >1GB (use LFS or S3)

""")